# Stage 1 TIMM-16 Baseline Comparison — ShrimpXNet-style

This standalone notebook runs the **16 TIMM/TorchVision-style lightweight models from the table**, excluding the ConvNeXt/ShrimpXNet reference by default.

Default model list:
- mobilenet_v3_large
- efficientnet_b0
- repvgg_a0
- efficientnet_v2_s
- shufflenet_v2_x1_0
- fastvit_t8
- edgenext_xx_small
- mobileone_s0
- mobilevit_s
- mobilenetv4_conv_small
- mnasnet_100
- ghostnetv2_100
- rexnet_100
- squeezenet1_1
- mobilenetv4_hybrid_medium
- efficientvit_m1

Optional:
- Set `INCLUDE_SHRIMPXNET_CONVNEXT = True` to also run `convnext_tiny_in22k` as the ShrimpXNet-style reference row.

Protocol:
- Local dataset only, no KaggleHub.
- 4 classes: Healthy, BG, WSSV, WSSV_BG.
- Stratified 70/15/15 split.
- CE baseline only.
- No RandAugment.
- No custom loss.
- No attention module.
- Saves CSV/JSON/XLSX/Markdown result tables and per-model artifacts.


In [ ]:

import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
DATASET_ROOT = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "datasets" / "processed-images"))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "runs")) / "stage1_timm16_shrimpxnet_outputs"

SEED = 42
IMG_SIZE = 224
EPOCHS = 30
PATIENCE = 5

# ShrimpXNet-style effective batch 128. Use micro-batch 32 for stability on notebooks/RTX.
PAPER_BATCH_SIZE = 128
MICRO_BATCH_SIZE = 32
ACCUMULATION_STEPS = max(1, PAPER_BATCH_SIZE // MICRO_BATCH_SIZE)
EVAL_BATCH_SIZE = 128

# Safer in notebooks/VS Code; set to 2 or 4 only after stable.
NUM_WORKERS = 0

LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_BACKBONE = 2e-5
LEARNING_RATE_FINETUNE_HEAD = 1e-4
STEP_SIZE = 3
STEP_GAMMA = 0.9
WARMUP_EPOCHS = 5

INCLUDE_SHRIMPXNET_CONVNEXT = False  # Set True for 17-row table including convnext_tiny_in22k.

CLASS_FOLDERS = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
CLASS_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

TIMM_MODELS_16 = [
    "mobilenet_v3_large",
    "efficientnet_b0",
    "repvgg_a0",
    "efficientnet_v2_s",
    "shufflenet_v2_x1_0",
    "fastvit_t8",
    "edgenext_xx_small",
    "mobileone_s0",
    "mobilevit_s",
    "mobilenetv4_conv_small",
    "mnasnet_100",
    "ghostnetv2_100",
    "rexnet_100",
    "squeezenet1_1",
    "mobilenetv4_hybrid_medium",
    "efficientvit_m1",
]

MODEL_KEYS = (["convnext_tiny_in22k"] if INCLUDE_SHRIMPXNET_CONVNEXT else []) + TIMM_MODELS_16

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)
print("Model count:", len(MODEL_KEYS))
print(MODEL_KEYS)


In [ ]:

import importlib, subprocess, sys, platform, json, os
from pathlib import Path

def has_module(name):
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "tqdm": "tqdm",
    "openpyxl": "openpyxl",
}

missing = [pkg for mod, pkg in required.items() if not has_module(mod)]
print("Missing packages:", missing)

# Do NOT reinstall torch automatically. Install only missing non-torch packages.
installable = [p for p in missing if p not in {"torch", "torchvision"}]
if installable:
    print("Installing only missing packages:", installable)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *installable])

import torch, torchvision, timm
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("TIMM:", timm.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for sub in ["manifests", "runs", "reports", "predictions", "confusion_matrices", "classification_reports", "logs"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

env = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "dataset_root": str(DATASET_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "seed": SEED,
    "img_size": IMG_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "paper_batch_size": PAPER_BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "accumulation_steps": ACCUMULATION_STEPS,
    "num_workers": NUM_WORKERS,
    "include_shrimpxnet_convnext": INCLUDE_SHRIMPXNET_CONVNEXT,
    "model_keys": MODEL_KEYS,
    "protocol": "TIMM-16 ShrimpXNet-style CE baseline; no RandAugment, no custom loss, no attention.",
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(env, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:

import random, hashlib, json, time, traceback, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image

def seed_everything(seed=SEED):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For reproducibility; benchmark can be faster but slightly nondeterministic.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

def md5_file(path: Path, chunk=1024 * 1024):
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def validate_dataset():
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"DATASET_ROOT does not exist: {DATASET_ROOT}")
    rows = []
    unreadable = []
    for label, folder in enumerate(CLASS_FOLDERS):
        d = DATASET_ROOT / folder
        if not d.exists():
            raise FileNotFoundError(f"Missing class folder: {d}")
        files = sorted([p for p in d.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS])
        print(f"{folder}: {len(files)}")
        for p in files:
            try:
                with Image.open(p) as im:
                    im.verify()
            except Exception as exc:
                unreadable.append({"path": str(p), "error": repr(exc)})
                continue
            rows.append({
                "source_path": str(p),
                "rel_path": str(p.relative_to(DATASET_ROOT)),
                "class_folder": folder,
                "class_name": CLASS_NAMES[label],
                "label": label,
                "file_size": p.stat().st_size,
                "md5": md5_file(p),
            })
    df = pd.DataFrame(rows)
    audit = {
        "dataset_root": str(DATASET_ROOT),
        "total_readable_images": int(len(df)),
        "unreadable_count": len(unreadable),
        "unreadable": unreadable[:50],
        "class_counts": df["class_name"].value_counts().to_dict() if not df.empty else {},
    }
    write_json(OUTPUT_DIR / "dataset_audit.json", audit)
    df.to_csv(OUTPUT_DIR / "manifests" / "source_manifest.csv", index=False)
    if len(df) != 1149:
        raise RuntimeError(f"Expected 1149 readable images, got {len(df)}. Check dataset path.")
    if unreadable:
        raise RuntimeError(f"Found unreadable images: {len(unreadable)}")
    return df

def make_split(df):
    split_path = OUTPUT_DIR / "manifests" / "split_manifest.csv"
    if split_path.exists():
        split_df = pd.read_csv(split_path)
        print("Using existing split:", split_path, split_df.shape)
        return split_df

    idx = np.arange(len(df))
    y = df["label"].values

    train_idx, temp_idx, _, y_temp = train_test_split(
        idx, y, train_size=0.70, random_state=SEED, stratify=y, shuffle=True
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=SEED, stratify=y_temp, shuffle=True
    )

    parts = []
    for split, ids in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        tmp = df.iloc[ids].copy()
        tmp["split"] = split
        parts.append(tmp)
    split_df = pd.concat(parts, ignore_index=True)
    split_df.to_csv(split_path, index=False)

    dist = split_df.groupby(["split", "class_name"]).size().reset_index(name="count")
    dist.to_csv(OUTPUT_DIR / "manifests" / "class_distribution.csv", index=False)
    print(dist.to_string(index=False))

    assert split_df.groupby("source_path")["split"].nunique().max() == 1, "Path overlap across splits"
    assert split_df.groupby("md5")["split"].nunique().max() == 1, "MD5 overlap across splits"
    return split_df

source_df = validate_dataset()
split_df = make_split(source_df)
print("Split sizes:", split_df["split"].value_counts().to_dict())


In [ ]:

import time, traceback, json, gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, cohen_kappa_score,
    confusion_matrix, classification_report,
)
from tqdm.auto import tqdm
import timm

TIMM_NAME_MAP = {
    "convnext_tiny_in22k": ["convnext_tiny.fb_in22k", "convnext_tiny.in12k_ft_in1k", "convnext_tiny"],
    "mobilenet_v3_large": ["mobilenetv3_large_100", "mobilenetv3_large_100.ra_in1k"],
    "efficientnet_b0": ["efficientnet_b0", "tf_efficientnet_b0"],
    "repvgg_a0": ["repvgg_a0"],
    "efficientnet_v2_s": ["tf_efficientnetv2_s", "efficientnetv2_rw_s"],
    "shufflenet_v2_x1_0": ["shufflenet_v2_x1_0"],
    "fastvit_t8": ["fastvit_t8.apple_in1k", "fastvit_t8"],
    "edgenext_xx_small": ["edgenext_xx_small"],
    "mobileone_s0": ["mobileone_s0.apple_in1k", "mobileone_s0"],
    "mobilevit_s": ["mobilevit_s"],
    "mobilenetv4_conv_small": ["mobilenetv4_conv_small.e2400_r224_in1k", "mobilenetv4_conv_small"],
    "mnasnet_100": ["mnasnet_100"],
    "ghostnetv2_100": ["ghostnetv2_100", "ghostnet_100"],
    "rexnet_100": ["rexnet_100"],
    "squeezenet1_1": ["squeezenet1_1"],
    "mobilenetv4_hybrid_medium": ["mobilenetv4_hybrid_medium.e500_r224_in1k", "mobilenetv4_hybrid_medium"],
    "efficientvit_m1": ["efficientvit_m1.r224_in1k", "efficientvit_m1"],
}

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def build_transforms():
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.90, 1.10),
                                     interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10, interpolation=InterpolationMode.BICUBIC),
        transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(236, interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tf, eval_tf

class ShrimpDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["source_path"]).convert("RGB")
        img = self.transform(img)
        label = int(row["label"])
        return img, label, row["source_path"]

def make_loaders(split_df):
    train_tf, eval_tf = build_transforms()
    train_ds = ShrimpDataset(split_df[split_df["split"] == "train"], train_tf)
    val_ds = ShrimpDataset(split_df[split_df["split"] == "val"], eval_tf)
    test_ds = ShrimpDataset(split_df[split_df["split"] == "test"], eval_tf)
    train_loader = DataLoader(train_ds, batch_size=MICRO_BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

def resolve_timm_name(model_key):
    tried = []
    last = ""
    for name in TIMM_NAME_MAP.get(model_key, [model_key]):
        tried.append(name)
        try:
            model = timm.create_model(name, pretrained=True, num_classes=4)
            model.eval()
            with torch.no_grad():
                out = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE))
            if list(out.shape) != [1, 4]:
                raise RuntimeError(f"bad output shape {tuple(out.shape)}")
            del model
            gc.collect()
            return name, ""
        except Exception as exc:
            last = repr(exc)
    return None, f"Tried {tried}; last_error={last}"

def create_model(model_key, device):
    name, reason = resolve_timm_name(model_key)
    if name is None:
        raise RuntimeError(reason)
    model = timm.create_model(name, pretrained=True, num_classes=4)
    return model.to(device), name

def set_trainable(model, phase):
    for p in model.parameters():
        p.requires_grad = False
    if phase == "head":
        for name, p in model.named_parameters():
            if any(k in name.lower() for k in ["classifier", "head", "fc"]):
                p.requires_grad = True
    elif phase == "finetune":
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(phase)

def make_optimizer(model, phase):
    if phase == "head":
        return torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE_HEAD)
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(k in name.lower() for k in ["classifier", "head", "fc"]):
            head_params.append(p)
        else:
            backbone_params.append(p)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LEARNING_RATE_BACKBONE})
    if head_params:
        groups.append({"params": head_params, "lr": LEARNING_RATE_FINETUNE_HEAD})
    return torch.optim.Adam(groups)

def eval_model(model, loader, criterion, device):
    model.eval()
    loss_sum, total, correct = 0.0, 0, 0
    y_true, y_pred, probs_all, paths = [], [], [], []
    with torch.no_grad():
        for x, y, p in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            probs = torch.softmax(logits, dim=1)
            pred = probs.argmax(1)
            loss_sum += float(loss.item())
            total += int(y.numel())
            correct += int((pred == y).sum().item())
            y_true.extend(y.detach().cpu().numpy().tolist())
            y_pred.extend(pred.detach().cpu().numpy().tolist())
            probs_all.extend(probs.detach().cpu().numpy().tolist())
            paths.extend(list(p))
    return loss_sum / max(1, len(loader)), correct / max(1, total), np.array(y_true), np.array(y_pred), np.array(probs_all), paths

def save_eval_outputs(model_key, split_name, y_true, y_pred, probs, paths, elapsed_s):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(4)), average="macro", zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred, labels=list(range(4)))
    pred_df = pd.DataFrame({
        "source_path": paths,
        "true_label": y_true,
        "true_class": [CLASS_NAMES[int(i)] for i in y_true],
        "pred_label": y_pred,
        "pred_class": [CLASS_NAMES[int(i)] for i in y_pred],
        "confidence": probs.max(axis=1),
    })
    for i, name in enumerate(CLASS_NAMES):
        pred_df[f"prob_{name}"] = probs[:, i]
    pred_df.to_csv(OUTPUT_DIR / "predictions" / f"{model_key}_{split_name}_predictions.csv", index=False)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(4)))
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        OUTPUT_DIR / "confusion_matrices" / f"{model_key}_{split_name}_confusion_matrix.csv"
    )
    report = classification_report(y_true, y_pred, labels=list(range(4)), target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(
        OUTPUT_DIR / "classification_reports" / f"{model_key}_{split_name}_classification_report.csv"
    )
    metrics = {
        "accuracy": float(acc),
        "macro_precision": float(pr),
        "macro_recall": float(rc),
        "macro_f1": float(f1),
        "kappa": float(kappa),
        "inference_time_s": float(elapsed_s),
        "latency_ms": float(elapsed_s / len(y_true) * 1000),
        "fps": float(len(y_true) / elapsed_s),
        "rows": int(len(y_true)),
    }
    write_json(OUTPUT_DIR / "reports" / f"{model_key}_{split_name}_metrics.json", metrics)
    return metrics

train_loader, val_loader, test_loader = make_loaders(split_df)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:

def train_one(model_key):
    run_dir = OUTPUT_DIR / "runs" / model_key
    metrics_path = run_dir / "metrics.json"
    if metrics_path.exists():
        print("SKIP completed:", model_key)
        return json.loads(metrics_path.read_text(encoding="utf-8"))

    run_dir.mkdir(parents=True, exist_ok=True)
    write_json(run_dir / "status.json", {"status": "started", "model": model_key, "time": time.ctime()})

    try:
        model, resolved_name = create_model(model_key, device)
        criterion = torch.nn.CrossEntropyLoss()

        history = []
        best_state = None
        best_val_loss = float("inf")
        best_epoch = -1
        patience_counter = 0
        total_start = time.perf_counter()

        phases = [("head", 0, min(WARMUP_EPOCHS, EPOCHS))]
        if EPOCHS > WARMUP_EPOCHS:
            phases.append(("finetune", WARMUP_EPOCHS, EPOCHS))

        for phase, start_epoch, end_epoch in phases:
            set_trainable(model, phase)
            optimizer = make_optimizer(model, phase)
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=STEP_GAMMA)

            for epoch in range(start_epoch, end_epoch):
                model.train()
                optimizer.zero_grad(set_to_none=True)
                loss_sum, correct, total = 0.0, 0, 0
                ep_start = time.perf_counter()

                pbar = tqdm(train_loader, desc=f"{model_key} epoch {epoch+1}/{EPOCHS} {phase}", leave=False)
                for step, (x, y, _) in enumerate(pbar):
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    logits = model(x)
                    loss = criterion(logits, y)
                    (loss / ACCUMULATION_STEPS).backward()

                    if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)

                    pred = logits.argmax(1)
                    loss_sum += float(loss.item())
                    correct += int((pred == y).sum().item())
                    total += int(y.numel())
                    pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/max(1,total):.3f}")

                scheduler.step()

                val_loss, val_acc, yv, pv, _, _ = eval_model(model, val_loader, criterion, device)
                _, _, val_f1, _ = precision_recall_fscore_support(yv, pv, labels=list(range(4)), average="macro", zero_division=0)

                row = {
                    "epoch": epoch + 1,
                    "phase": phase,
                    "train_loss": loss_sum / max(1, len(train_loader)),
                    "train_accuracy": correct / max(1, total),
                    "val_loss": float(val_loss),
                    "val_accuracy": float(val_acc),
                    "val_macro_f1": float(val_f1),
                    "epoch_time_s": float(time.perf_counter() - ep_start),
                    "lr": float(optimizer.param_groups[0]["lr"]),
                }
                history.append(row)
                print(model_key, row)

                if not np.isfinite(val_loss):
                    raise RuntimeError(f"Non-finite val_loss: {val_loss}")

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_epoch = epoch + 1
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    torch.save(best_state, run_dir / "best.pt")
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= PATIENCE:
                        print("Early stopping:", model_key, "epoch", epoch + 1)
                        break

        if best_state is None or not (run_dir / "best.pt").exists():
            raise RuntimeError("No best checkpoint was saved")

        model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device))

        val_start = time.perf_counter()
        _, _, yv, pv, probv, pathv = eval_model(model, val_loader, criterion, device)
        val_metrics = save_eval_outputs(model_key, "val", yv, pv, probv, pathv, time.perf_counter() - val_start)

        test_start = time.perf_counter()
        _, _, yt, pt, probt, patht = eval_model(model, test_loader, criterion, device)
        test_metrics = save_eval_outputs(model_key, "test", yt, pt, probt, patht, time.perf_counter() - test_start)

        pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

        out = {
            "model": model_key,
            "resolved_name": resolved_name,
            "status": "completed",
            "parameters_m": float(sum(p.numel() for p in model.parameters()) / 1e6),
            "model_size_mb": float((run_dir / "best.pt").stat().st_size / (1024 * 1024)),
            "training_time_s": float(time.perf_counter() - total_start),
            "best_epoch": int(best_epoch),
            "checkpoint_path": str(run_dir / "best.pt"),
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_kappa": val_metrics["kappa"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_precision": test_metrics["macro_precision"],
            "test_macro_recall": test_metrics["macro_recall"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_kappa": test_metrics["kappa"],
            "inference_time_s": test_metrics["inference_time_s"],
            "latency_ms": test_metrics["latency_ms"],
            "fps": test_metrics["fps"],
            "loss": "baseline_ce",
            "randaugment": False,
            "attention": "none",
            "protocol": "TIMM-16 ShrimpXNet-style baseline",
        }
        write_json(metrics_path, out)
        write_json(run_dir / "status.json", {"status": "completed", "model": model_key, "time": time.ctime()})

        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        gc.collect()
        return out

    except Exception as exc:
        tb = traceback.format_exc()
        write_json(run_dir / "status.json", {"status": "failed", "model": model_key, "error": repr(exc), "traceback": tb})
        print("FAILED:", model_key, repr(exc))
        return {"model": model_key, "status": "failed", "failure_reason": repr(exc)}

# Availability check
availability = []
for m in MODEL_KEYS:
    name, reason = resolve_timm_name(m)
    availability.append({"model": m, "available": name is not None, "resolved_name": name or "", "reason": reason})
availability_df = pd.DataFrame(availability)
availability_df.to_csv(OUTPUT_DIR / "reports" / "model_availability_timm.csv", index=False)
display(availability_df)

rows = []
for m in MODEL_KEYS:
    if not availability_df.loc[availability_df["model"] == m, "available"].iloc[0]:
        rows.append({"model": m, "status": "skipped_unavailable", "failure_reason": availability_df.loc[availability_df["model"] == m, "reason"].iloc[0]})
        continue
    rows.append(train_one(m))

raw = pd.DataFrame(rows)
raw.to_csv(OUTPUT_DIR / "final_timm_baseline_comparison_raw.csv", index=False)

completed = raw[raw["status"] == "completed"].copy()
failed = raw[raw["status"] != "completed"].copy()

if not completed.empty:
    completed = completed.sort_values(
        ["test_macro_f1", "test_kappa", "test_accuracy", "latency_ms", "model_size_mb"],
        ascending=[False, False, False, True, True],
    )
    completed.to_csv(OUTPUT_DIR / "final_timm_baseline_ranking.csv", index=False)
    completed.to_json(OUTPUT_DIR / "final_timm_baseline_ranking.json", orient="records", indent=2)
    completed.to_excel(OUTPUT_DIR / "final_timm_baseline_ranking.xlsx", index=False)
    (OUTPUT_DIR / "final_timm_baseline_ranking.md").write_text(
        "# Stage 1 TIMM-16 Baseline Ranking\n\n" + completed.to_markdown(index=False),
        encoding="utf-8",
    )
    display(completed[["model", "resolved_name", "test_macro_f1", "test_accuracy", "test_kappa", "latency_ms", "fps", "model_size_mb"]])

failed.to_csv(OUTPUT_DIR / "failed_or_skipped_timm_models.csv", index=False)
write_json(OUTPUT_DIR / "final_run_summary.json", {
    "planned": len(MODEL_KEYS),
    "completed": int(len(completed)),
    "failed_or_skipped": int(len(failed)),
    "include_shrimpxnet_convnext": INCLUDE_SHRIMPXNET_CONVNEXT,
    "output_dir": str(OUTPUT_DIR),
})
print("DONE:", OUTPUT_DIR)


In [ ]:

from pathlib import Path
import pandas as pd, json, zipfile

out = OUTPUT_DIR
for p in [
    out / "reports" / "model_availability_timm.csv",
    out / "final_timm_baseline_ranking.csv",
    out / "failed_or_skipped_timm_models.csv",
    out / "final_run_summary.json",
]:
    print("\n==", p, "exists", p.exists(), "size", p.stat().st_size if p.exists() else None)
    if p.exists() and p.suffix == ".csv":
        df = pd.read_csv(p)
        print(df.shape)
        display(df.head(30))
    elif p.exists():
        print(p.read_text(encoding="utf-8")[:4000])

zip_path = PROJECT_ROOT / "stage1_timm16_shrimpxnet_for_review.zip"
if zip_path.exists():
    zip_path.unlink()

def add_file(zf, p, root, written):
    if not p.is_file():
        return
    if p.suffix.lower() in {".pt", ".pth", ".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
        return
    if "__pycache__" in p.parts or "weights" in p.parts:
        return
    if p.stat().st_size > 20_000_000:
        return
    try:
        arc = p.relative_to(root).as_posix()
    except Exception:
        arc = p.name
    if arc in written:
        return
    zf.write(p, arc)
    written.add(arc)

written = set()
root = PROJECT_ROOT
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    nb_path = PROJECT_ROOT / "notebooks" / "stage1_timm16_shrimpxnet_baseline_reference.ipynb"
    if nb_path.exists():
        add_file(zf, nb_path, root, written)
    for p in out.rglob("*"):
        add_file(zf, p, root, written)

print("Created ZIP:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 3))
